In [ ]:
import pyarrow.parquet as pq
import os
import pandas as pd

In [ ]:
base_path = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line"
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"

materials_file = os.path.join(base_path, "materials_single_line.parquet")
filtered_output_file = os.path.join(output_dir, "filtered_materials.parquet")
final_output_file = os.path.join(output_dir, "filtered_materials_encoded.parquet")

## Step 1: Filter columns with PyArrow

In [ ]:
# Columns to retain
keep_columns = [
    "component_position", "component_id", "serial_number_id",
    "station_id", "supplier_id", "mounting_place",
    "container_number", "panel_position", "created_at"
]

In [ ]:
# Load the full Parquet file
table = pq.read_table(materials_file)

# Drop columns not in the keep list
columns_to_drop = [col for col in table.column_names if col not in keep_columns]
filtered_table = table.drop(columns_to_drop)

In [ ]:
# Save the filtered table first (without encoding yet)
pq.write_table(filtered_table, filtered_output_file)
print(f"Filtered materials saved to: {filtered_output_file}")

 ## Step 2: Encode the container_number column

In [ ]:
# Reload the smaller, filtered file for encoding
df = pd.read_parquet(filtered_output_file)

In [ ]:
# Frequency encode 'container_number'
container_freq = df['container_number'].value_counts()
df['container_number_freq'] = df['container_number'].map(container_freq)

#  Drop the original 'container_number' to reduce memory usage
df.drop(columns=['container_number'], inplace=True)

In [ ]:
#Save the filtered table
df.to_parquet(final_output_file, index=False)
print(f"Filtered and encoded materials saved to: {final_output_file}")

In [ ]:
df.head()

## Analyse columns

In [ ]:
# Load filtered table into Pandas for analysis
df = filtered_table.to_pandas()

In [ ]:
# Columns to inspect for sparsity and cardinality
cols_to_analyze = ['container_number', 'supplier_id', 'supplier_order_id', 'component_id']

In [ ]:
# Calculate % missing and unique counts
missing_pct = df[cols_to_analyze].isna().mean() * 100
unique_counts = df[cols_to_analyze].nunique()

In [ ]:
# Value counts for top 10 values (optional, detailed)
value_counts = {}
for col in cols_to_analyze:
    value_counts[col] = df[col].value_counts().head(10)

# Step 8: Summary table
summary = pd.DataFrame({
    '% Missing': missing_pct,
    'Unique Values': unique_counts
})

In [ ]:
print("\n=== Summary of Sparsity & Cardinality ===\n")
print(summary)

print("\n=== Top 10 Frequent Values for Each Column ===\n")
for col, counts in value_counts.items():
    print(f"\nColumn: {col}\n")
    print(counts)

In [ ]:
# How many 1-to-1 mappings?
correlation_check = df.groupby('container_number')['supplier_order_id'].nunique()
one_to_one_ratio = (correlation_check == 1).mean()
print(f"% of container_numbers that map to exactly 1 supplier_order_id: {one_to_one_ratio * 100:.2f}%")

drop supplier_order_id, keep supplier_id component_id, frequency encode container_number